# سبق 03 - ایجنٹک ڈیزائن پیٹرنز

اس سبق میں، ہم موثر AI ایجنٹس بنانے کے لیے تین بنیادی ڈیزائن پیٹرنز کا جائزہ لیتے ہیں:

1. **صاف ایجنٹ ہدایات** — واضح، کردار کی تعریف کرنے والے پرامپٹس تیار کرنا جو ایجنٹ کے رویے کی رہنمائی کرتے ہیں
2. **پائیڈینٹک ماڈلز کے ساتھ منظم آؤٹ پٹ** — اس بات کو یقینی بنانا کہ ایجنٹس پیش گوئی کے قابل اور توثیق شدہ ڈیٹا واپس کریں
3. **سنگل ریسپانسبلٹی ایجنٹس** — توجہ مرکوز ایجنٹس کو ڈیزائن کرنا جو ہر ایک چیز کو اچھی طرح انجام دے

ہم ہر پیٹرن کو **سفر کی منزل کی سفارش کرنے والے** منظرنامے پر لاگو کریں گے، ایک ایسا نظام مرحلہ وار بناتے ہوئے جو مقامات کی تجویز دے سکے، دستیابی چیک کر سکے، اور لاجسٹکس سنبھال سکے۔


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## نمونہ 1: ایجنٹ کے لیے واضح ہدایات

سب سے مؤثر نمونہ سب سے سادہ بھی ہے: اپنے ایجنٹ کے لیے واضح، مفصل ہدایات لکھنا۔

اچھی ہدایات یہ تعین کرتی ہیں:
- **کون** ایجنٹ ہے (کردار اور لہجہ)
- **کیا** اسے کرنا چاہیے (مرحلہ وار ذمہ داریاں)
- **کیسے** اسے برتاؤ کرنا چاہیے (محدودیات اور انداز)

نیچے، ہم ایک سفری کنسیئر ایجنٹ بناتے ہیں جس کی واضح ہدایات ہر جواب کو تشکیل دیتی ہیں جو یہ پیدا کرتا ہے۔


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

The key to enforcement is passing `response_format` when we run the agent. This forces the
model to return a validated `TravelRecommendations` object (available on `response.value`)
instead of free-form text. The `get_destination_details` tool also returns a typed
`DestinationRecommendation`, so the data stays structured end to end.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## نمونہ 3: واحد ذمہ داری والے ایجنٹس

پیچیدہ کاموں میں کام کو کئی مخصوص ایجنٹس میں تقسیم کرنے سے فائدہ ہوتا ہے، ہر ایک کی ایک واحد ذمہ داری ہوتی ہے:

- ایک **مقصد کے ماہر** جو جگہوں اور دستیابی کے بارے میں جانتا ہے
- ایک **لاجسٹکس منصوبہ ساز** جو پروازوں، ہوٹلوں، اور سفرناموں کو سنبھالتا ہے

یہ سافٹ ویئر انجینیئرنگ کے اصول *علاقوں کی تفریق* کی مانند ہے — ہر ایجنٹ کو آزادانہ طور پر آزمانا، برقرار رکھنا، اور بہتر بنانا آسان ہوتا ہے۔


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصہ

اس سبق میں ہم نے ایک سفر کی سفارش کرنے والے منظرنامے پر تین ایجنٹک ڈیزائن پیٹرنز کا اطلاق کیا:

| پیٹرن | کلیدی خیال | فائدہ |
|---|---|---|
| **واضح ہدایات** | شخصیت، ذمہ داریوں، اور پابندیوں کو پہلے سے متعین کریں | مستقل، برانڈ کے مطابق ایجنٹ کا رویہ |
| **منظم آؤٹ پٹ** | جوابی فارمیٹ کے طور پر Pydantic ماڈلز کا استعمال کریں | تصدیق شدہ، مشین پڑھنے کے قابل نتائج |
| **واحد ذمہ داری** | ہر ایجنٹ کو ایک مخصوص کام دیں | جانچنے، برقرار رکھنے، اور ملاپ کرنے میں آسان |

یہ پیٹرنز قدرتی طور پر ایک ساتھ مل کر کام کرتے ہیں — آپ واضح ہدایات کو منظم آؤٹ پٹ کے ساتھ اور واحد ذمہ داری والے ایجنٹ میں ملا کر مضبوط، پیداواری نظام تیار کر سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
